In [3]:
!pip install pennylane
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import load_wine # WINE VERİ SETİ İÇİN EKLENDİ
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================
# 1. VERİ HAZIRLIĞI VE SIZINTISIZ BÖLME
# ==========================================
print("--- WINE VERİ SETİ (3 SINIFLI) ANN VE QNN (5-FOLD CV) DENEYLERİ BAŞLATILIYOR ---")

# WINE VERİ SETİNİ YÜKLEME
wine = load_wine()
X_raw = wine.data
y_raw = wine.target
input_dim = X_raw.shape[1]

print(f"Toplam Örneklem: {len(X_raw)} (Sınıflar: {np.bincount(y_raw)})")
print(f"Girdi Boyutu: {input_dim}\n")

# ==========================================
# 2. MODEL TANIMLARI
# ==========================================
def create_hybrid_qnn_class(n_qubits, n_layers=2):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 3)}

    class DynamicHybridQNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 16)
            self.dropout = nn.Dropout(0.3)
            self.fc_bottleneck = nn.Linear(16, n_qubits)
            self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
            # 3 Sınıflı olduğu için çıkış 3 nöron (Sigmoid kaldırıldı, CrossEntropy kendi halledecek)
            self.fc_out = nn.Linear(n_qubits, 3)

        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = self.dropout(x)
            x = torch.relu(self.fc_bottleneck(x))
            x = torch.tanh(x) * np.pi
            x = self.qlayer(x)
            x = self.fc_out(x)
            return x

    return DynamicHybridQNN

class PureClassicalANN(nn.Module):
    def __init__(self, bottleneck_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(16, bottleneck_dim)
        self.fc3 = nn.Linear(bottleneck_dim, bottleneck_dim)
        # 3 Sınıflı olduğu için çıkış 3 nöron
        self.fc_out = nn.Linear(bottleneck_dim, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc_out(x)
        return x

# ==========================================
# 3. 5-FOLD ÇAPRAZ DOĞRULAMA MOTORU
# ==========================================
def evaluate_model_kfold(model_builder_fn, model_name, epochs=30, lr=0.002, batch_size=32):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    fold_acc = []
    fold_f1_weighted = [] # 3 sınıf için binary f1 yerine weighted
    fold_f1_macro = []
    fold_auc = []

    print(f"--- [{model_name}] 5-Fold Çapraz Doğrulama Başlatılıyor ({epochs} Epoch) ---")
    start_total_t = time.time()

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw), 1):
        X_tr_raw, X_va_raw = X_raw[train_idx], X_raw[val_idx]
        y_tr_np, y_va_np = y_raw[train_idx], y_raw[val_idx]

        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_tr_scaled = scaler.fit_transform(X_tr_raw)
        X_va_scaled = scaler.transform(X_va_raw)

        # Hedefler int/long olmalı (CrossEntropy için)
        train_ds = TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32),
                                 torch.tensor(y_tr_np, dtype=torch.long))
        val_ds = TensorDataset(torch.tensor(X_va_scaled, dtype=torch.float32),
                               torch.tensor(y_va_np, dtype=torch.long))

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        model = model_builder_fn()
        criterion = nn.CrossEntropyLoss() # 3 sınıf için BCE yerine CrossEntropy
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                optimizer.zero_grad()
                out = model(bx)
                loss = criterion(out, by) # unsqueeze kaldırıldı
                loss.backward()
                optimizer.step()

        model.eval()
        y_true, y_pred, y_probs = [], [], []
        with torch.no_grad():
            for bx, by in val_loader:
                out = model(bx)
                preds = torch.argmax(out, dim=1) # 0.5 threshold yerine argmax
                probs = F.softmax(out, dim=1)    # AUC için olasılıklar

                y_true.extend(by.numpy())
                y_pred.extend(preds.numpy().flatten())
                y_probs.extend(probs.numpy())

        acc = accuracy_score(y_true, y_pred)
        f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        f1_m = f1_score(y_true, y_pred, average="macro", zero_division=0)
        # Çoklu sınıf için OVR stratejisi
        auc = roc_auc_score(y_true, np.array(y_probs), multi_class="ovr")

        fold_acc.append(acc)
        fold_f1_weighted.append(f1_w)
        fold_f1_macro.append(f1_m)
        fold_auc.append(auc)
        print(f"   Fold {fold}/5: Acc: %{acc*100:.2f} | F1: %{f1_w*100:.2f} | AUC: {auc:.3f}")

    total_elapsed = time.time() - start_total_t
    print(f"[{model_name}] Tamamlandı (Toplam Süre: {total_elapsed:.1f} sn)\n")

    return {
        "acc_mean": np.mean(fold_acc) * 100,
        "acc_std": np.std(fold_acc) * 100,
        "f1_weighted_mean": np.mean(fold_f1_weighted) * 100,
        "f1_weighted_std": np.std(fold_f1_weighted) * 100,
        "f1_macro_mean": np.mean(fold_f1_macro) * 100,
        "f1_macro_std": np.std(fold_f1_macro) * 100,
        "auc_mean": np.mean(fold_auc),
        "auc_std": np.std(fold_auc),
        "total_time": total_elapsed
    }

# ==========================================
# 4. MODELLERİN 5-FOLD CV ÇALIŞTIRILMASI
# ==========================================
cv_summary = {}

QNN2_Builder = create_hybrid_qnn_class(n_qubits=2, n_layers=2)
cv_summary["Hibrit QNN (2 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN2_Builder, "Hibrit QNN (2 Kübit)", epochs=30)

QNN4_Builder = create_hybrid_qnn_class(n_qubits=4, n_layers=2)
cv_summary["Hibrit QNN (4 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN4_Builder, "Hibrit QNN (4 Kübit)", epochs=30)

QNN8_Builder = create_hybrid_qnn_class(n_qubits=8, n_layers=2)
cv_summary["Hibrit QNN (8 Kübit, 30 Ep)"] = evaluate_model_kfold(QNN8_Builder, "Hibrit QNN (8 Kübit)", epochs=30)

ANN_Builder = lambda: PureClassicalANN(bottleneck_dim=2)
cv_summary["Klasik ANN (0 Kübit, 30 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (30 Epoch)", epochs=30)
cv_summary["Klasik ANN (0 Kübit, 100 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (100 Epoch)", epochs=100)
cv_summary["Klasik ANN (0 Kübit, 200 Ep)"] = evaluate_model_kfold(ANN_Builder, "Klasik ANN (200 Epoch)", epochs=200)

# ==========================================
# 5. HAKEM #2 YORUM #4 İÇİN SONUÇ TABLOSU
# ==========================================
print("\n" + "=" * 110)
print(f"{'Model Mimarisi':<32} | {'Doğruluk (Mean ± SD)':<22} | {'F1 Skoru (Mean ± SD)':<22} | {'ROC-AUC':<16} | {'Süre'}")
print("=" * 110)
for name, stats in cv_summary.items():
    acc_str = f"%{stats['acc_mean']:.2f} ± {stats['acc_std']:.2f}"
    f1_str = f"%{stats['f1_weighted_mean']:.2f} ± {stats['f1_weighted_std']:.2f}"
    auc_str = f"{stats['auc_mean']:.3f} ± {stats['auc_std']:.3f}"
    print(f"{name:<32} | {acc_str:<22} | {f1_str:<22} | {auc_str:<16} | {stats['total_time']:.1f} sn")
print("=" * 110)

--- WINE VERİ SETİ (3 SINIFLI) ANN VE QNN (5-FOLD CV) DENEYLERİ BAŞLATILIYOR ---
Toplam Örneklem: 178 (Sınıflar: [59 71 48])
Girdi Boyutu: 13

--- [Hibrit QNN (2 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5: Acc: %69.44 | F1: %59.80 | AUC: 0.835
   Fold 2/5: Acc: %94.44 | F1: %94.44 | AUC: 0.976
   Fold 3/5: Acc: %72.22 | F1: %61.99 | AUC: 0.837
   Fold 4/5: Acc: %94.29 | F1: %94.29 | AUC: 0.993
   Fold 5/5: Acc: %74.29 | F1: %65.16 | AUC: 0.866
[Hibrit QNN (2 Kübit)] Tamamlandı (Toplam Süre: 14.5 sn)

--- [Hibrit QNN (4 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch) ---
   Fold 1/5: Acc: %94.44 | F1: %94.45 | AUC: 0.998
   Fold 2/5: Acc: %94.44 | F1: %94.44 | AUC: 0.984
   Fold 3/5: Acc: %91.67 | F1: %91.70 | AUC: 1.000
   Fold 4/5: Acc: %71.43 | F1: %61.80 | AUC: 0.868
   Fold 5/5: Acc: %74.29 | F1: %64.40 | AUC: 1.000
[Hibrit QNN (4 Kübit)] Tamamlandı (Toplam Süre: 17.1 sn)

--- [Hibrit QNN (8 Kübit)] 5-Fold Çapraz Doğrulama Başlatılıyor (30 Epoch